# Capability audit: what Selene can actually be asked

`project_scoping.md` asserts a machine surface — what is exposed, what is hidden, what can
be measured. Those assertions were written before any of it was run. This notebook tests
each one and records a verdict, so the rescope rests on evidence rather than expectation.

**Headline: the contention claims cannot be validated here.** Selene has no zone model, no
transport model, and no duration term in any error model. Two of the four checks in §8 are
unanswerable rather than merely unanswered. The erasure-herald claim, by contrast, holds up
fully.

| Verdict | Meaning |
| --- | --- |
| **CONFIRMED** | the scoping document is right |
| **REFUTED** | the scoping document is wrong and needs changing |
| **UNANSWERABLE** | the question cannot be put to this stack at all |

## 0. Setup

In [1]:
import inspect, os, subprocess
import pandas as pd

import selene_sim
import selene_core.runtime
from guppylang import guppy
from guppylang.std import qsystem
from guppylang.std.builtins import output, py
from guppylang.std.quantum import qubit, h, x, measure
from guppylang.std.qsystem import measure_leaked, measure_and_reset
from selene_sim import (DepolarizingErrorModel, IdealErrorModel, SimpleLeakageErrorModel,
                        SimpleRuntime, HeliosInterface, Quest, build)

SITE = os.path.dirname(os.path.dirname(selene_sim.__file__))
pd.set_option('display.width', 130)
pd.set_option('display.max_colwidth', 70)
verdicts = []


def record(claim, verdict, evidence):
    verdicts.append({'claim': claim, 'verdict': verdict, 'evidence': evidence})

## 1. What is installed

The plugin list is the whole surface. Anything absent here is absent, full stop — these are
compiled shared objects, so a missing capability means writing and building a plugin, not
passing a different argument.

In [2]:
plugins = sorted(d for d in os.listdir(SITE)
                 if d.startswith('selene_') and not d.endswith('info'))
groups = {'runtime': [], 'error model': [], 'QIS / interface': [], 'simulator': [], 'other': []}
for p in plugins:
    if 'runtime' in p:            groups['runtime'].append(p)
    elif 'error_model' in p:      groups['error model'].append(p)
    elif 'qis' in p:              groups['QIS / interface'].append(p)
    elif any(s in p for s in ('quest', 'stim')): groups['simulator'].append(p)
    else:                          groups['other'].append(p)

for k, v in groups.items():
    print(f'{k:18} {", ".join(x.replace("selene_", "").replace("_plugin", "") for x in v)}')

print('\nNo plugin provides zones, transport, shuttling, or a Wasm decoder.')

runtime            simple_runtime, soft_rz_runtime
error model        depolarizing_error_model, ideal_error_model, simple_leakage_error_model
QIS / interface    base_qis, helios_qis, hugr_qis_compiler, sol_qis
simulator          quest, stim
other              argreader, classical_replay, coinflip, core, envreader, quantum_replay, sim

No plugin provides zones, transport, shuttling, or a Wasm decoder.


In [3]:
# Grep the whole Python stack for the concepts the scoping document depends on.
targets = ['zone', 'transport', 'shuttl', 'wasm', 'byoc', 'amplitude_damp', 'dephas', 't1']
rows = []
for term in targets:
    r = subprocess.run(['grep', '-rilE', term, f'{SITE}/selene_sim', f'{SITE}/selene_core'],
                       capture_output=True, text=True)
    files = [f for f in r.stdout.strip().splitlines() if f and '__pycache__' not in f]
    rows.append({'term': term, 'files': len(files),
                 'where': ', '.join(f.split('site-packages/')[-1] for f in files[:2]) or '-'})
pd.DataFrame(rows).set_index('term')

,files,where
term,,
zone,0,-
transport,1,selene_core/runtime.py
shuttl,0,-
wasm,0,-
byoc,0,-
amplitude_damp,0,-
dephas,0,-
t1,5,"selene_sim/event_hooks/instruction_log.py, selene_sim/_dist/lib/li..."


`transport` appears only in a docstring, and it is worth reading exactly what it says.

In [4]:
doc = inspect.getdoc(selene_core.runtime.Runtime)
i = doc.find('In the specific context')
print(doc[i:i + 760])

In the specific context of ion trap quantum computers, there are other things
that a runtime may wish to perform, such as routing ions, deciding on optimal
order of gates to minimise transport or error rates, and so on. While Selene
itself does not understand such commands, it provides a message channel from
the runtime to the error model through an opaque "custom operation" type. Through
this mechanism, a runtime designed for specific hardware can 'talk' to an error
model designed for the same hardware, allowing, for example:
- noise to accumulated through transport
- noise to be dependent on relative locations of qubits
- specific timing information to be taken into account


So transport modelling is **architecturally possible and practically absent**. A custom
runtime plugin could emit opaque custom operations that a matching custom error model
interprets as transport noise. Both are compiled shared objects, neither exists, and
writing them is a project rather than a parameter.

That is the difference between "hidden" — the scoping document's word — and what is
actually the case: **not modelled at all**.

In [5]:
record('§2 Native gates (phased_x, rz, zz_phase, zz_max) exposed', 'CONFIRMED',
       f'guppylang.std.qsystem exports: {", ".join(sorted(n for n in dir(qsystem) if n in ("phased_x","rz","zz_phase","zz_max")))}')
record('§2 Placement / transport / batching hidden behind the runtime', 'REFUTED',
       'Not hidden — absent. No zone or transport model in any installed plugin; '
       'Runtime ABC offers only an opaque channel that nothing implements.')
record('§2 A decoder can be written and run', 'CONFIRMED',
       'Python post-processing always; and in-program in real time, since '
       'measurement-conditioned control flow works (section 5).')
record('§2 Decoder latency is first-class (Wasm BYOC over a network)', 'REFUTED',
       'No wasm or byoc path in the stack, and no duration feeds any error model. '
       'Decoding works; its latency cannot be measured.')
record('§2 Logical-qubit abstraction absent', 'CONFIRMED',
       'Encoded circuits are hand-written; no logical type in guppylang.std.')
print(f'{len(verdicts)} verdicts so far')

5 verdicts so far


## 2. §8 check 1 — is the noise model phenomenological?

The highest-impact question in the plan. Three error models exist; the test is whether any
of them charges for *time* rather than for *operations*.

In [6]:
for M in (IdealErrorModel, DepolarizingErrorModel, SimpleLeakageErrorModel):
    print(f'{M.__name__:28}{inspect.signature(M.__init__)}')
print(f'\n{"SimpleRuntime":28}{inspect.signature(SimpleRuntime.__init__)}')
print('\nEvery error-model parameter attaches to an operation. Duration appears only in')
print('the runtime, which is a separate object and feeds scheduling rather than error.')

IdealPlugin                 (self, random_seed: int | None = None) -> None
DepolarizingPlugin          (self, random_seed: int | None = None, p_1q: float = 0.0, p_2q: float = 0.0, p_meas: float = 0.0, p_init: float = 0.0) -> None
SimpleLeakagePlugin         (self, random_seed: int | None = None, p_leak: float = 0.0, leak_measurement_bias: float = 0.0) -> None

SimpleRuntime               (self, random_seed: int | None = None, duration_ns_rxy: int = 0, duration_ns_rzz: int = 0, duration_ns_rz: int = 0, duration_ns_rpp: int = 0, duration_ns_measure: int = 0, duration_ns_reset: int = 0, duration_ns_measure_leaked: int = 0) -> None

Every error-model parameter attaches to an operation. Duration appears only in
the runtime, which is a separate object and feeds scheduling rather than error.


In [7]:
# A qubit allocated, left alone, and measured. If idling cost anything, this would show it.
@guppy
def untouched() -> None:
    q = qubit()
    output('m', measure(q).read())


runner = build(untouched.compile())
rows = []
for label, em in [('ideal', IdealErrorModel(random_seed=1)),
                  ('gate channels at 0.3', DepolarizingErrorModel(random_seed=1, p_1q=0.3, p_2q=0.3)),
                  ('leakage at 0.3', SimpleLeakageErrorModel(random_seed=1, p_leak=0.3))]:
    shots = [dict(s) for s in runner.run_shots(simulator=Quest(), n_qubits=2, n_shots=600,
                                               error_model=em, random_seed=1)]
    rows.append({'error model': label, 'P(flip on an idle qubit)': sum(s['m'] for s in shots) / 600})

record('§8.1 Noise model is phenomenological (no timing layer)', 'CONFIRMED',
       'Three models, all operation-attached. Idling is free unless written as explicit ops.')
pd.DataFrame(rows).set_index('error model')

,P(flip on an idle qubit)
error model,
ideal,0.0
gate channels at 0.3,0.0
leakage at 0.3,0.0


An untouched qubit stays perfect under every model. **Idling costs nothing until it is
written into the circuit as operations**, which makes the idling rate a declared parameter
rather than a measured one.

This is what makes §8 checks 2 and 3 unanswerable.

## 3. §8 checks 2 and 3 — zone sharing and transport amortisation

Check 2 asks whether syndrome rounds and algorithm gates share zones. Check 3 asks whether
transport amortises across a 16-ion batch.

Neither can be put to this stack: there is no zone, no ion, no transport, and no duration
that feeds error. The plan already anticipates this outcome — §3 of `action_plan.md` says a
phenomenological result at B2 "makes this element unanswerable in Selene… record both
quantities as declared model assumptions and skip the element."

In [8]:
record('§8.2 Do syndrome rounds and algorithm gates share zones?', 'UNANSWERABLE',
       'No zone model exists in any installed plugin.')
record('§8.3 Does transport amortise across a batch?', 'UNANSWERABLE',
       'No transport model. Runtime ABC describes an opaque channel; no plugin implements it.')
record('§8.4 Is the reachable d-range wide enough to test 1/d scaling?', 'REFUTED',
       'Steane admits d=3 only. Testing 1/d needs a distance-parameterised family.')
print('checks 2 and 3 recorded as unanswerable')

checks 2 and 3 recorded as unanswerable


## 4. B3 — do leakage heralds work?

The one gate in the plan that could still go either way, and the load-bearing one for the
erasure-herald claim. `measure_leaked` returns `MaybeLeaked`; `is_leaked()` is the herald.

The question is whether heralds arrive **at a rate under experimental control**.

In [9]:
def leak_probe(n_ops):
    @guppy
    def prog() -> None:
        q = qubit()
        for _ in range(py(n_ops)):
            h(q)
        m = measure_leaked(q)
        output('leaked', m.is_leaked())
        m.discard()
    return prog


rows = []
for p_leak in (0.0, 0.01, 0.05, 0.1, 0.3):
    row = {'p_leak': p_leak}
    for n_ops in (1, 10):
        r = build(leak_probe(n_ops).compile())
        em = (IdealErrorModel(random_seed=1) if p_leak == 0
              else SimpleLeakageErrorModel(random_seed=1, p_leak=p_leak))
        shots = [dict(s) for s in r.run_shots(simulator=Quest(), n_qubits=4, n_shots=800,
                                              error_model=em, random_seed=1)]
        row[f'P(herald), {n_ops} ops'] = round(sum(s['leaked'] for s in shots) / 800, 3)
    rows.append(row)

record('§9/C2 Leakage heralds available at a controllable rate (gate B3)', 'CONFIRMED',
       'P(herald) scales with both p_leak and operation count; 0 at p_leak=0.')
pd.DataFrame(rows).set_index('p_leak')

,"P(herald), 1 ops","P(herald), 10 ops"
p_leak,,
0.00,0.000,0.000
0.01,0.015,0.158
0.05,0.086,0.618
0.10,0.194,0.873
0.30,0.530,0.999


In [10]:
# Does leak_measurement_bias change the herald rate?
rows = []
for bias in (0.0, 0.5, 1.0):
    r = build(leak_probe(10).compile())
    em = SimpleLeakageErrorModel(random_seed=1, p_leak=0.1, leak_measurement_bias=bias)
    shots = [dict(s) for s in r.run_shots(simulator=Quest(), n_qubits=4, n_shots=800,
                                          error_model=em, random_seed=1)]
    rows.append({'leak_measurement_bias': bias,
                 'P(herald)': round(sum(s['leaked'] for s in shots) / 800, 3)})
print('leak_measurement_bias does not move the herald rate — it biases the value a leaked')
print('qubit reports when measured normally, not whether measure_leaked flags it.\n')
pd.DataFrame(rows).set_index('leak_measurement_bias')

leak_measurement_bias does not move the herald rate — it biases the value a leaked
qubit reports when measured normally, not whether measure_leaked flags it.



,P(herald)
leak_measurement_bias,
0.0,0.873
0.5,0.873
1.0,0.873


## 5. B5 — real-time classical control

Adaptive policies need a branch on a measurement result, and ancilla reuse needs
`measure_and_reset`. Both are claimed in §2.

In [11]:
@guppy
def feedback() -> None:
    a, b = qubit(), qubit()
    h(a)
    m = measure(a).read()
    if m:            # branch on a measurement, mid-circuit
        x(b)
    output('a', m)
    output('b', measure(b).read())


r = build(feedback.compile())
shots = [dict(s) for s in r.run_shots(simulator=Quest(), n_qubits=4, n_shots=400,
                                      error_model=IdealErrorModel(random_seed=1), random_seed=1)]
agree = sum(1 for s in shots if s['a'] == s['b'])


@guppy
def reuse() -> None:
    a = qubit()
    total = 0
    for _ in range(8):
        h(a)
        if measure_and_reset(a).read():    # same physical qubit, eight times
            total += 1
    output('ones', total)
    measure(a)


r2 = build(reuse.compile())
vals = [dict(s)['ones'] for s in r2.run_shots(simulator=Quest(), n_qubits=4, n_shots=400,
                                              error_model=IdealErrorModel(random_seed=1),
                                              random_seed=1)]
print(f'measurement-conditioned branch: b == a in {agree}/400 shots')
print(f'measure_and_reset x8 in a loop:  mean ones = {sum(vals)/len(vals):.2f} of 8 '
      f'(range {min(vals)}-{max(vals)})')

record('§2 Real-time classical control flow and RNG exposed', 'CONFIRMED',
       f'Measurement-conditioned branch correct in {agree}/400 shots; '
       'measure_and_reset works in a loop.')

measurement-conditioned branch: b == a in 400/400 shots
measure_and_reset x8 in a loop:  mean ones = 3.92 of 8 (range 0-8)


## 6. Verdicts

In [12]:
audit = pd.DataFrame(verdicts)
counts = audit.verdict.value_counts()
print(counts.to_string(), '\n')
audit.set_index('claim')

verdict
CONFIRMED       6
REFUTED         3
UNANSWERABLE    2 



,verdict,evidence
claim,,
"§2 Native gates (phased_x, rz, zz_phase, zz_max) exposed",CONFIRMED,"guppylang.std.qsystem exports: phased_x, rz, zz_max, zz_phase"
§2 Placement / transport / batching hidden behind the runtime,REFUTED,Not hidden — absent. No zone or transport model in any installed p...
§2 A decoder can be written and run,CONFIRMED,"Python post-processing always; and in-program in real time, since ..."
§2 Decoder latency is first-class (Wasm BYOC over a network),REFUTED,"No wasm or byoc path in the stack, and no duration feeds any error..."
§2 Logical-qubit abstraction absent,CONFIRMED,Encoded circuits are hand-written; no logical type in guppylang.std.
§8.1 Noise model is phenomenological (no timing layer),CONFIRMED,"Three models, all operation-attached. Idling is free unless writte..."
§8.2 Do syndrome rounds and algorithm gates share zones?,UNANSWERABLE,No zone model exists in any installed plugin.
§8.3 Does transport amortise across a batch?,UNANSWERABLE,No transport model. Runtime ABC describes an opaque channel; no pl...
§8.4 Is the reachable d-range wide enough to test 1/d scaling?,REFUTED,Steane admits d=3 only. Testing 1/d needs a distance-parameterised...


## 7. What the scoping document needs

**§2 machine surface.** Two rows are wrong.

- *Placement, transport, batching: "Hidden — runtime, real-time, per-branch"* → these are
  **absent**, not hidden. Nothing models them and nothing can be asked about them.
- *Decoder: "Exposed — Wasm BYOC or fixed BP-OSD/GPU; both over network | Latency is
  first-class"* → split this row. **Decoding is fully available**: in Python
  post-processing, and in-program in real time, since measurement-conditioned control flow
  works. A Steane lookup-table decoder running inside the Guppy program is straightforward,
  and that is what an adaptive τ policy needs. What fails is **latency**: there is no Wasm
  path, no network round-trip, and no duration term anywhere, so decoder service time
  cannot be measured. C3 loses its mechanism; the decoder itself is untouched.

**§8 checks.** Check 1 resolves phenomenological. Checks 2 and 3 are therefore unanswerable
and should be recorded as declared assumptions, per the action plan's own rule. Check 4
resolves against the current code family: Steane is d=3 only.

**§4 contributions.** C1's contended-zone-time mechanism and C3's decoder *latency* both
rest on quantities this stack cannot produce. They survive only as declared models, which
§7 must state as a primary limitation rather than a caveat.

**C2 survives intact.** Heralds work, at a controllable rate, and herald-aware versus
herald-blind decoding is measurable. It is the strongest remaining claim.

### The honest v1

> The optimal syndrome interval is set by the fault rate of the extraction circuit, which is
> chosen rather than given; erasure heralds move it further. Zone contention enters as a
> declared cost model, not a measurement.

`τ` also needs restating: with no duration in any error model, idle cycles cost nothing
unless inserted as explicit operations, so τ's cost is a parameter you set rather than one
the simulator charges. `noise_analysis.ipynb` measures the idle operation that makes this
work.

### Next

1. Rewrite §2, §8 and §7 of `project_scoping.md` against this table.
2. Mark B2 and B3 closed in `action_plan.md`; mark B4 skipped.
3. Record the A2 decision — existence of an interior optimum, dropping 1/d scaling.